# MH6301 — Yelp Business Search Engine (Improved)

**IR Library:** Whoosh (BM25F) + Sentence-Transformers (dense embeddings)  
**Document definition:** Each document = one Philadelphia business, with all its reviews concatenated into one `reviews` field.  
**Linguistic processing:**
- Case folding ✓ (Whoosh default)
- Stemming ✓ (Whoosh `StemmingAnalyzer` via Porter stemmer)
- Stopword removal ✓ (NLTK english stopwords, 179 terms)
- Tokenisation ✓ (whitespace + punctuation split)

**Searches supported:**
1. Keyword search on business name (BM25F, fuzzy-tolerant)
2. Similar-review search — BM25F over extracted keywords
3. Similar-review search — cosine similarity over `all-MiniLM-L6-v2` embeddings
4. Hybrid similar-review search — linear combination of BM25 + embedding scores
5. Incremental index update (add new businesses/reviews without full rebuild)

## Cell 1 — Imports

In [1]:
# Install spell-correction library (run once)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'symspellpy', '-q'], check=True)
print('symspellpy ready')

symspellpy ready


In [2]:
import os
import re
import shutil
import time
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Whoosh ────────────────────────────────────────────────────────────────────
from whoosh import index
from whoosh.fields import Schema, TEXT, ID, NUMERIC, STORED
from whoosh.analysis import StemmingAnalyzer, Filter, LowercaseFilter, StopFilter
from whoosh.qparser import MultifieldParser, FuzzyTermPlugin
from whoosh.query import Term
from whoosh.scoring import BM25F

# ── NLP / Embeddings ──────────────────────────────────────────────────────────
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Spell correction ──────────────────────────────────────────────────────────
try:
    from symspellpy import SymSpell, Verbosity
    _SYMSPELL_OK = True
except ImportError:
    _SYMSPELL_OK = False
    print("symspellpy not installed — run: pip install symspellpy")

# ── Kaggle dataset ─────────────────────────────────────────────────────────────
import kagglehub

print("All imports OK")

All imports OK


## Cell 2 — Load & prepare data

In [3]:
# Download dataset
path = kagglehub.dataset_download("waterwaterf/yelp-review-business")
print("Dataset path:", path)

# Load raw CSVs
df_business = pd.read_csv(os.path.join(path, "business.csv"))
df_review   = pd.read_csv(os.path.join(path, "review_light (1).csv"))

# ── Filter to Philadelphia ────────────────────────────────────────────────────
df_biz_philly = df_business[df_business["city"] == "Philadelphia"].copy()
df_rev_philly = df_review[df_review["business_id"].isin(df_biz_philly["business_id"])].copy()

print(f"Philadelphia businesses : {len(df_biz_philly):,}")
print(f"Philadelphia reviews    : {len(df_rev_philly):,}")

# ── Aggregate reviews per business ───────────────────────────────────────────
review_grouped = (
    df_rev_philly
    .groupby("business_id")["text"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"text": "reviews"})
)

# ── Merge ─────────────────────────────────────────────────────────────────────
df_index = df_biz_philly.merge(review_grouped, on="business_id", how="left")
df_index["reviews"] = df_index["reviews"].fillna("")

# Normalise column names (handle both 'name' and 'business_name')
if "business_name" in df_index.columns:
    df_index = df_index.rename(columns={"business_name": "name"})

# Ensure numeric columns exist
for col, default in [("stars", 0.0), ("review_count", 0)]:
    if col not in df_index.columns:
        df_index[col] = default

df_index = df_index.reset_index(drop=True)
print("\nMerged df_index shape:", df_index.shape)
df_index[["business_id", "name", "city", "stars", "review_count"]].head()

Dataset path: /Users/reguluskai/.cache/kagglehub/datasets/waterwaterf/yelp-review-business/versions/1
Philadelphia businesses : 14,569
Philadelphia reviews    : 967,552

Merged df_index shape: (14569, 11)


,business_id,name,city,stars,review_count
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,0.0,0
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,Philadelphia,0.0,0
2,ROeacJQwBeh05Rqg7F6TCg,BAP,Philadelphia,0.0,0
3,QdN72BWoyFypdGJhhI5r7g,Bar One,Philadelphia,0.0,0
4,Mjboz24M9NlBeiOJKLEd_Q,DeSandro on Main,Philadelphia,0.0,0


## Cell 3 — Schema & Index construction

**Document definition:** One Whoosh document per business. Fields:
| Field | Type | Analysed | Stored | Purpose |
|-------|------|----------|--------|---------|
| `business_id` | ID | No | Yes | Unique lookup key |
| `name` | TEXT | Yes | Yes | Keyword search on business name |
| `city` | TEXT | No | Yes | Display |
| `categories` | TEXT | Yes | Yes | Keyword search on categories |
| `reviews` | TEXT | Yes | Yes | Similar-review BM25 search + snippets |
| `stars` | NUMERIC | No | Yes | Display |
| `review_count` | NUMERIC | No | Yes | Display |
| `address` | STORED | No | Yes | Display only |

**Analyser:** `StemmingAnalyzer` which applies lowercase → tokenise → stopword removal (Whoosh built-in) → Porter stemming.

In [4]:
INDEX_DIR        = "philly_index"
EMBEDDINGS_PATH  = "philly_embeddings.pkl"

# Analyser: lowercase + tokenise + stop-word removal + Porter stemming
analyzer = StemmingAnalyzer()

schema = Schema(
    business_id  = ID(stored=True, unique=True),
    name         = TEXT(stored=True, analyzer=analyzer),
    city         = TEXT(stored=True),
    categories   = TEXT(stored=True, analyzer=analyzer),
    reviews      = TEXT(stored=True, analyzer=analyzer),
    stars        = NUMERIC(stored=True, numtype=float, default=0.0),
    review_count = NUMERIC(stored=True, numtype=int,   default=0),
    address      = STORED(),
)

# ── (Re)build index ───────────────────────────────────────────────────────────
if os.path.exists(INDEX_DIR):
    shutil.rmtree(INDEX_DIR)
os.mkdir(INDEX_DIR)

ix = index.create_in(INDEX_DIR, schema)

# Use a BufferedWriter for speed
writer = ix.writer(limitmb=256)

print("Indexing documents …")
t0 = time.time()

for _, row in tqdm(df_index.iterrows(), total=len(df_index)):
    writer.add_document(
        business_id  = str(row["business_id"]),
        name         = str(row["name"]),
        city         = str(row.get("city", "")),
        categories   = str(row.get("categories", "")),
        reviews      = str(row["reviews"]),
        stars        = float(row.get("stars", 0.0)),
        review_count = int(row.get("review_count", 0)),
        address      = str(row.get("address", "")),
    )

writer.commit()
print(f"Index built in {time.time()-t0:.1f}s  |  {ix.doc_count()} documents")

Indexing documents …


100%|█████████████████████████████████████| 14569/14569 [04:16<00:00, 56.88it/s]


Index built in 365.3s  |  14569 documents


## Cell 4 — Build & cache dense embeddings

**Similarity definition:** We encode each business's aggregated review text with `all-MiniLM-L6-v2`  
(384-dim), then use **cosine similarity** as the similarity measure.  

**Why this reflects review similarity well:**  
Sentence-transformers are trained on hundreds of millions of sentence pairs to place semantically  
similar texts close together in the embedding space. A business where reviews mention "cosy atmosphere,  
great wine" will be near another business where reviews say "intimate vibe, excellent drinks" —  
despite zero lexical overlap. This captures the *semantic* meaning of reviews rather than just shared words.

In [5]:
print("Loading SentenceTransformer model …")
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading cached embeddings …")
    with open(EMBEDDINGS_PATH, "rb") as f:
        emb_data = pickle.load(f)
    embeddings      = emb_data["embeddings"]
    emb_business_ids = emb_data["business_ids"]
    print(f"Loaded {len(embeddings)} cached embeddings.")
else:
    print("Computing embeddings (this takes a few minutes) …")
    t0 = time.time()
    embeddings = emb_model.encode(
        df_index["reviews"].tolist(),
        show_progress_bar=True,
        batch_size=64,
    )
    emb_business_ids = df_index["business_id"].tolist()
    with open(EMBEDDINGS_PATH, "wb") as f:
        pickle.dump({"embeddings": embeddings,
                     "business_ids": emb_business_ids}, f)
    print(f"Embeddings computed & cached in {time.time()-t0:.1f}s")

# Build a fast id→index lookup
emb_id_to_idx = {bid: i for i, bid in enumerate(emb_business_ids)}
print(f"Embedding matrix shape: {embeddings.shape}")

Loading SentenceTransformer model …


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached embeddings …
Loaded 14569 cached embeddings.
Embedding matrix shape: (14569, 384)


## Cell 5 — Helper utilities

In [6]:
# ── Stopwords (NLTK english, 179 terms) ──────────────────────────────────────
STOPWORDS = set(nltk_stopwords.words("english"))

def extract_keywords(text: str, top_k: int = 30) -> str:
    """
    Extract the top-k most frequent non-stopword words (≥3 chars) from text.
    Used to build a BM25 query for similar-business search.
    """
    words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
    words = [w for w in words if w not in STOPWORDS]
    freq  = Counter(words)
    return " ".join(w for w, _ in freq.most_common(top_k))


def _sep():
    print("─" * 60)


def _print_result(rank: int, score: float, hit, snippet: str = ""):
    """Unified result printer for BM25 hits."""
    print(f"\n  Rank {rank}  │  Score: {score:.4f}  │  DocID: {hit['business_id']}")
    print(f"  Business : {hit['name']}")
    print(f"  City     : {hit['city']}")
    print(f"  Stars    : {hit.get('stars', '?')}  │  Reviews: {hit.get('review_count', '?')}")
    print(f"  Category : {str(hit['categories'])[:80]}")
    if snippet:
        # strip HTML highlight tags for clean console output
        clean = re.sub(r"<[^>]+>", "**", snippet)
        print(f"  Snippet  : {clean[:250]}")
    _sep()


def _print_emb_result(rank: int, score: float, row: pd.Series):
    """Unified result printer for embedding hits."""
    print(f"\n  Rank {rank}  │  Cosine: {score:.4f}  │  DocID: {row['business_id']}")
    print(f"  Business : {row['name']}")
    print(f"  City     : {row['city']}")
    print(f"  Stars    : {row.get('stars', '?')}  │  Reviews: {row.get('review_count', '?')}")
    print(f"  Category : {str(row.get('categories', ''))[:80]}")
    # Snippet: first 200 chars of aggregated reviews
    snippet = str(row.get("reviews", ""))[:200].replace("\n", " ")
    if snippet:
        print(f"  Snippet  : {snippet} …")
    _sep()

print("Helpers ready.")

Helpers ready.


## Cell 6 — Search functions

In [7]:
# ── Open a persistent index handle ───────────────────────────────────────────
# (opened ONCE; kept open for the session — avoids per-call overhead)
_ix = index.open_dir(INDEX_DIR)


# ─────────────────────────────────────────────────────────────────────────────
# 0a.  SPELL CORRECTION  (SymSpell compound lookup)
# ─────────────────────────────────────────────────────────────────────────────
_sym = None

def _init_symspell() -> bool:
    """Lazy-load SymSpell with its bundled frequency dictionary."""
    global _sym
    if _sym is not None:
        return True
    if not _SYMSPELL_OK:
        return False
    import pkg_resources
    _sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
    dict_path = pkg_resources.resource_filename(
        "symspellpy", "frequency_dictionary_en_82_765.txt"
    )
    ok = _sym.load_dictionary(dict_path, term_index=0, count_index=1)
    if not ok:
        print("Warning: SymSpell dictionary failed to load.")
        _sym = None
        return False
    return True


def correct_spelling(query: str) -> str:
    """
    Return a spell-corrected version of query using SymSpell compound lookup.
    Prints a notice if any correction was made.
    Falls back to the original query when symspellpy is unavailable.
    """
    if not _init_symspell():
        return query
    suggestions = _sym.lookup_compound(query, max_edit_distance=2)
    if not suggestions:
        return query
    corrected = suggestions[0].term
    if corrected.lower() != query.lower():
        print(f"  Spell check: '{query}'  →  '{corrected}'")
    return corrected


# ─────────────────────────────────────────────────────────────────────────────
# 0b.  SYNONYM EXPANSION  (food & restaurant domain dictionary)
# ─────────────────────────────────────────────────────────────────────────────
SYNONYMS: dict[str, list[str]] = {
    # Drinks
    "coffee":      ["coffee", "cafe", "espresso", "latte", "cappuccino"],
    "tea":         ["tea", "boba", "bubble tea", "matcha"],
    "beer":        ["beer", "brewery", "craft beer", "ale", "lager"],
    "cocktail":    ["cocktail", "bar", "speakeasy", "mixology"],
    # Meats / proteins
    "burger":      ["burger", "burgers", "smash", "patty", "cheeseburger"],
    "chicken":     ["chicken", "poultry", "wings", "fried chicken"],
    "steak":       ["steak", "steakhouse", "beef", "ribeye", "sirloin"],
    "seafood":     ["seafood", "fish", "shrimp", "lobster", "crab", "oyster"],
    "sushi":       ["sushi", "sashimi", "maki", "omakase", "japanese"],
    # Cuisines
    "italian":     ["italian", "pasta", "pizza", "trattoria", "osteria"],
    "mexican":     ["mexican", "tacos", "burrito", "tex-mex", "enchilada"],
    "chinese":     ["chinese", "dim sum", "cantonese", "szechuan", "mandarin"],
    "indian":      ["indian", "curry", "tandoor", "masala", "naan"],
    "thai":        ["thai", "pad thai", "curry", "bangkok"],
    "korean":      ["korean", "kbbq", "bibimbap", "korean bbq"],
    "vietnamese":  ["vietnamese", "pho", "banh mi", "viet"],
    "mediterranean": ["mediterranean", "greek", "turkish", "lebanese", "falafel"],
    # Meal type
    "breakfast":   ["breakfast", "brunch", "eggs", "pancakes", "waffles"],
    "bbq":         ["bbq", "barbecue", "barbeque", "smokehouse", "smoked"],
    "pizza":       ["pizza", "pizzeria", "neapolitan", "slice"],
    "sandwich":    ["sandwich", "sub", "hoagie", "deli", "grinder"],
    "salad":       ["salad", "healthy", "greens", "bowl"],
    "dessert":     ["dessert", "cake", "ice cream", "gelato", "pastry", "bakery"],
    # Venue type
    "bar":         ["bar", "pub", "tavern", "lounge", "gastropub"],
    "cafe":        ["cafe", "coffee", "bakery", "bistro"],
    "restaurant":  ["restaurant", "eatery", "dining", "bistro", "grill"],
    "food truck":  ["food truck", "street food", "pop-up"],
    "buffet":      ["buffet", "all you can eat", "smorgasbord"],
    "vegan":       ["vegan", "plant-based", "vegetarian"],
}


def expand_query(query: str) -> str:
    """
    Replace known tokens with OR-grouped synonym clusters.
    E.g. 'burger bar'  →  '(burger OR burgers OR smash ...) (bar OR pub OR ...)'
    Multi-word synonyms are checked first (longest match wins).
    Returns the original query unchanged if no synonyms match.
    """
    lower = query.lower()
    result_tokens = []
    i = 0
    words = lower.split()
    used = set()

    while i < len(words):
        matched = False
        # Try two-word phrases first
        if i + 1 < len(words):
            bigram = words[i] + " " + words[i+1]
            if bigram in SYNONYMS:
                syns = SYNONYMS[bigram]
                result_tokens.append("(" + " OR ".join(syns) + ")")
                used.add(bigram)
                i += 2
                matched = True
        if not matched:
            tok = words[i]
            if tok in SYNONYMS:
                syns = SYNONYMS[tok]
                result_tokens.append("(" + " OR ".join(syns) + ")")
            else:
                result_tokens.append(tok)
            i += 1

    expanded = " ".join(result_tokens)
    if expanded != query:
        print(f"  Synonyms:    '{query}'  →  '{expanded}'")
    return expanded


# ─────────────────────────────────────────────────────────────────────────────
# 1.  KEYWORD SEARCH  (BM25F on name + categories, fuzzy-tolerant)
# ─────────────────────────────────────────────────────────────────────────────
def _auto_fuzzify(query: str, min_len: int = 4, max_edits: int = 1) -> str:
    """
    Append a fuzzy suffix (~N) to each token that is long enough to benefit
    from edit-distance matching (min_len chars).  Tokens that already carry
    a Whoosh operator (*, ?, ~, ", +, -) are left untouched.

    Example:  "coffe shop"  →  "coffe~1 shop~1"
              "Wi-Fi"       →  "Wi~1 Fi~1"
    """
    tokens = query.split()
    fuzzified = []
    for tok in tokens:
        # Leave tokens that already have special Whoosh syntax alone
        if any(c in tok for c in ('*', '?', '~', '"', '+', '-')):
            fuzzified.append(tok)
        elif len(tok) >= min_len:
            fuzzified.append(f"{tok}~{max_edits}")
        else:
            fuzzified.append(tok)
    return " ".join(fuzzified)


def keyword_search(query: str, top_n: int = 10) -> None:
    """
    Search businesses by keywords in their name and category fields.

    Processing pipeline (in order):
      1. Spell correction  — fix typos via SymSpell  (e.g. 'coffe' → 'coffee')
      2. Synonym expansion — broaden query with OR groups  (e.g. 'burger' →
         'burger OR burgers OR smash OR patty OR cheeseburger')
      3. BM25F search      — exact + stemmed match across name (×2) + categories
      4. Fuzzy fallback    — if still no results, auto-append ~1 to each token

    Scoring: Whoosh BM25F with k1=1.2, B=0.75, name boosted ×2.
    """
    # ── Pipeline step 1 & 2 ──────────────────────────────────────────────────
    query = correct_spelling(query)
    query = expand_query(query)

    with _ix.searcher(weighting=BM25F(B=0.75, name_B=0.5)) as searcher:
        parser = MultifieldParser(
            ["name", "categories"],
            schema=_ix.schema,
            fieldboosts={"name": 2.0, "categories": 1.0},
        )
        parser.add_plugin(FuzzyTermPlugin())

        fuzzy_used = False
        try:
            q = parser.parse(query)
        except Exception as e:
            print(f"Query parse error: {e}")
            return

        t0 = time.time()
        results = searcher.search(q, limit=top_n)

        # ── Fallback: auto-fuzzify if no results ─────────────────────────────
        if not results:
            fuzzy_query_str = _auto_fuzzify(query)
            if fuzzy_query_str != query:          # only retry if something changed
                try:
                    q_fuzzy = parser.parse(fuzzy_query_str)
                    results  = searcher.search(q_fuzzy, limit=top_n)
                    fuzzy_used = True
                except Exception:
                    pass                          # silently fall through to "no results"

        elapsed = time.time() - t0

        print(f"\n{'═'*60}")
        print(f"  BM25 KEYWORD SEARCH")
        print(f"  Query   : {query}" + (" (fuzzy fallback applied)" if fuzzy_used else ""))
        print(f"  Hits    : {len(results)} (of {results.estimated_length()} estimated)")
        print(f"  Time    : {elapsed*1000:.1f} ms")
        print(f"{'═'*60}")

        if not results:
            print("  No results found. Try a different query or check for typos.")
            return

        for i, r in enumerate(results, 1):
            snippet = r.highlights("name") or r.highlights("categories") or ""
            _print_result(i, r.score, r, snippet)


# ─────────────────────────────────────────────────────────────────────────────
# 2.  SIMILAR REVIEWS — BM25 keyword extraction approach
# ─────────────────────────────────────────────────────────────────────────────
def bm25_similar(business_id: str, top_n: int = 10) -> None:
    """
    Find businesses with reviews similar to the given business using BM25.

    Method:
    1. Retrieve the source business's aggregated reviews.
    2. Extract the top-30 most-frequent non-stopword terms (keyword profile).
    3. Issue a BM25 query on the 'reviews' field with those terms.
    4. Exclude the source business from results.

    Similarity definition: businesses whose review text shares the most
    salient topical terms with the source, weighted by BM25F TF-IDF.
    """
    with _ix.searcher() as searcher:
        # Lookup source business
        res = searcher.search(Term("business_id", business_id), limit=1)
        if not res:
            print(f"Business ID '{business_id}' not found in index.")
            return

        source      = res[0]
        query_text  = extract_keywords(source["reviews"], top_k=30)

        if not query_text.strip():
            print("Source business has no usable review text.")
            return

        parser = MultifieldParser(["reviews"], schema=_ix.schema)
        try:
            q = parser.parse(query_text)
        except Exception as e:
            print(f"Query parse error: {e}")
            return

        t0      = time.time()
        results = searcher.search(q, limit=top_n + 1)
        elapsed = time.time() - t0

        print(f"\n{'═'*60}")
        print(f"  BM25 SIMILAR BUSINESSES")
        print(f"  Source  : {source['name']}  (ID: {business_id})")
        print(f"  Keywords: {query_text[:80]} …")
        print(f"  Time    : {elapsed*1000:.1f} ms")
        print(f"{'═'*60}")

        rank = 1
        for r in results:
            if r["business_id"] == business_id:
                continue
            snippet = r.highlights("reviews") or ""
            _print_result(rank, r.score, r, snippet)
            rank += 1
            if rank > top_n:
                break


# ─────────────────────────────────────────────────────────────────────────────
# 3.  SIMILAR REVIEWS — Dense embedding (semantic) approach
# ─────────────────────────────────────────────────────────────────────────────
def embedding_similar(business_id: str, top_n: int = 10) -> None:
    """
    Find businesses with semantically similar reviews using cosine similarity
    over all-MiniLM-L6-v2 sentence embeddings.

    Similarity definition: the 384-dimensional embedding of a business's
    aggregated review text captures the latent semantic 'experience' of
    visiting that business. Cosine similarity between two embeddings
    measures how alike those experiences are — independent of exact wording.
    """
    idx = emb_id_to_idx.get(business_id)
    if idx is None:
        print(f"Business ID '{business_id}' not found in embedding index.")
        return

    t0   = time.time()
    sims = cosine_similarity(embeddings[idx].reshape(1, -1), embeddings)[0]
    top  = np.argsort(-sims)[1:top_n + 1]   # skip rank-0 (self)
    elapsed = time.time() - t0

    source_name = df_index.loc[idx, "name"]

    print(f"\n{'═'*60}")
    print(f"  EMBEDDING SIMILAR BUSINESSES")
    print(f"  Source  : {source_name}  (ID: {business_id})")
    print(f"  Time    : {elapsed*1000:.1f} ms")
    print(f"{'═'*60}")

    for rank, i in enumerate(top, 1):
        _print_emb_result(rank, float(sims[i]), df_index.loc[i])


# ─────────────────────────────────────────────────────────────────────────────
# 4.  HYBRID SIMILAR REVIEWS  (BM25 normalised score + embedding cosine)
# ─────────────────────────────────────────────────────────────────────────────
def hybrid_similar(business_id: str, top_n: int = 10,
                   alpha: float = 0.5) -> None:
    """
    Combines BM25 keyword-based similarity and embedding semantic similarity
    into a single ranked list.

    final_score = alpha * norm_bm25_score + (1 - alpha) * cosine_similarity

    alpha=0.0  → pure embedding
    alpha=0.5  → equal weight (default)
    alpha=1.0  → pure BM25

    Both scores are min-max normalised across all candidates before combining.
    """
    # ── Validate alpha ───────────────────────────────────────────────────────
    if not (0.0 <= alpha <= 1.0):
        clamped = max(0.0, min(1.0, alpha))
        print(f"  Warning: alpha={alpha} out of range [0, 1] — clamped to {clamped}.")
        alpha = clamped

    # ── Embedding scores for all documents ──────────────────────────────────
    idx = emb_id_to_idx.get(business_id)
    if idx is None:
        print(f"Business ID '{business_id}' not found.")
        return

    emb_sims = cosine_similarity(embeddings[idx].reshape(1, -1), embeddings)[0]

    # ── BM25 scores (top 200 candidates) ────────────────────────────────────
    bm25_scores = {}
    with _ix.searcher() as searcher:
        res = searcher.search(Term("business_id", business_id), limit=1)
        if not res:
            print(f"Business ID '{business_id}' not found in BM25 index.")
            return

        source     = res[0]
        source_name = source["name"]
        query_text = extract_keywords(source["reviews"], top_k=30)

        if query_text.strip():
            parser  = MultifieldParser(["reviews"], schema=_ix.schema)
            q       = parser.parse(query_text)
            results = searcher.search(q, limit=200)
            for r in results:
                bm25_scores[r["business_id"]] = r.score

    # ── Merge & normalise ─────────────────────────────────────────────────────
    # Use len(embeddings) as the canonical size — emb_sims has this length.
    # df_index may differ if add_or_update_business() added rows to the
    # embedding matrix without a matching df_index row (or vice-versa).
    n_docs   = len(emb_business_ids)
    bm25_arr = np.zeros(n_docs)
    for bid, score in bm25_scores.items():
        emb_pos = emb_id_to_idx.get(bid)
        if emb_pos is not None:
            bm25_arr[emb_pos] = score

    def minmax(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)

    norm_bm25 = minmax(bm25_arr)   # shape (n_docs,)
    norm_emb  = minmax(emb_sims)   # shape (n_docs,)  — guaranteed same length
    hybrid    = alpha * norm_bm25 + (1 - alpha) * norm_emb

    # Exclude source
    hybrid[idx] = -1
    top = np.argsort(-hybrid)[: top_n]

    t_label = f"α={alpha:.2f}  (BM25×{alpha:.2f} + Emb×{1-alpha:.2f})"

    print(f"\n{'═'*60}")
    print(f"  HYBRID SIMILAR BUSINESSES")
    print(f"  Source  : {source_name}  (ID: {business_id})")
    print(f"  Mix     : {t_label}")
    print(f"{'═'*60}")

    for rank, i in enumerate(top, 1):
        bid_i = emb_business_ids[i]
        # Look up display info from df_index; fall back gracefully if missing
        rows = df_index[df_index["business_id"] == bid_i]
        if rows.empty:
            print(f"\n  Rank {rank}  │  Hybrid: {hybrid[i]:.4f}  "
                  f"(BM25ₙ={norm_bm25[i]:.3f}, Embₙ={norm_emb[i]:.3f})  "
                  f"│  DocID: {bid_i}")
            print(f"  Business : (details not in df_index)")
            _sep()
            continue
        row = rows.iloc[0]
        print(f"\n  Rank {rank}  │  Hybrid: {hybrid[i]:.4f}  "
              f"(BM25ₙ={norm_bm25[i]:.3f}, Embₙ={norm_emb[i]:.3f})  "
              f"│  DocID: {row['business_id']}")
        print(f"  Business : {row['name']}")
        print(f"  Stars    : {row.get('stars', '?')}  │  Reviews: {row.get('review_count', '?')}")
        print(f"  Category : {str(row.get('categories', ''))[:80]}")
        _sep()


print("All search functions defined.")

All search functions defined.


## Cell 7 — Incremental index update (online data handling)

**Discussion on online data readiness:**  
The current architecture handles incremental updates as follows:
- Whoosh's `writer.update_document()` adds a new document or overwrites an existing one by `business_id` (the unique key), so new reviews can be appended without a full rebuild.
- The embedding cache is updated by appending the new embedding vector and saving the pickle.
- For very high-throughput streams (e.g. thousands of reviews/second), the architecture would benefit from a dedicated message queue (Kafka) feeding a background indexing worker, with periodic Whoosh `commit()` batches.

In [8]:
def add_or_update_business(
    business_id: str,
    name: str,
    city: str,
    categories: str,
    new_reviews: str,
    stars: float = 0.0,
    review_count: int = 0,
    address: str = "",
) -> None:
    """
    Add a new business OR append new reviews to an existing one.
    Updates both the Whoosh BM25 index and the embedding cache.

    This function simulates handling an online stream of new reviews:
    each call is O(1) — no full index rebuild required.
    """
    global embeddings, emb_business_ids, emb_id_to_idx, df_index

    # ── 1. Update Whoosh index ────────────────────────────────────────────────
    # If business exists, retrieve existing reviews and concatenate
    existing_reviews = ""
    with _ix.searcher() as searcher:
        res = searcher.search(Term("business_id", business_id), limit=1)
        if res:
            existing_reviews = res[0]["reviews"]
            print(f"Updating existing business: {res[0]['name']}")
        else:
            print(f"Adding new business: {name}")

    combined_reviews = (existing_reviews + " " + new_reviews).strip()

    writer = _ix.writer()
    writer.update_document(
        business_id  = business_id,
        name         = name,
        city         = city,
        categories   = categories,
        reviews      = combined_reviews,
        stars        = stars,
        review_count = review_count,
        address      = address,
    )
    writer.commit()

    # ── 2. Update df_index ────────────────────────────────────────────────────
    existing_row = df_index[df_index["business_id"] == business_id]
    if len(existing_row):
        df_index.loc[existing_row.index[0], "reviews"] = combined_reviews
    else:
        new_row = pd.DataFrame([{
            "business_id": business_id, "name": name, "city": city,
            "categories": categories, "reviews": combined_reviews,
            "stars": stars, "review_count": review_count, "address": address,
        }])
        df_index = pd.concat([df_index, new_row], ignore_index=True)

    # ── 3. Recompute embedding for this business ──────────────────────────────
    new_emb = emb_model.encode([combined_reviews])[0]

    if business_id in emb_id_to_idx:
        i = emb_id_to_idx[business_id]
        embeddings[i] = new_emb
    else:
        i = len(emb_business_ids)
        embeddings         = np.vstack([embeddings, new_emb])
        emb_business_ids.append(business_id)
        emb_id_to_idx[business_id] = i

    # ── 4. Persist updated embeddings ─────────────────────────────────────────
    with open(EMBEDDINGS_PATH, "wb") as f:
        pickle.dump({"embeddings": embeddings,
                     "business_ids": emb_business_ids}, f)

    print(f"✓ Index and embeddings updated for business_id={business_id}")
    print(f"  Total documents: {_ix.doc_count()} BM25  |  {len(emb_business_ids)} embeddings")


print("add_or_update_business() ready.")

add_or_update_business() ready.


## Cell 8 — Interactive console loop

In [9]:
MENU = """
╔══════════════════════════════════════════╗
║       YELP PHILADELPHIA SEARCH ENGINE   ║
╠══════════════════════════════════════════╣
║  1. Keyword search (BM25F + fuzzy)      ║
║  2. Similar reviews – BM25 keywords     ║
║  3. Similar reviews – Embeddings        ║
║  4. Similar reviews – Hybrid            ║
║  5. Add / update business (online sim)  ║
║  6. Exit                                ║
╚══════════════════════════════════════════╝"""


def _get_int(prompt: str, minimum: int = 1) -> int | None:
    try:
        n = int(input(prompt))
        if n < minimum:
            print(f"  Must be ≥ {minimum}.")
            return None
        return n
    except ValueError:
        print("  Invalid number.")
        return None


def _get_float(prompt: str, lo: float = 0.0, hi: float = 1.0) -> float | None:
    try:
        v = float(input(prompt))
        if not (lo <= v <= hi):
            print(f"  Must be between {lo} and {hi}.")
            return None
        return v
    except ValueError:
        print("  Invalid number.")
        return None


print(MENU)

while True:
    choice = input("\nChoice [1-6]: ").strip()

    # ── 1. Keyword search ─────────────────────────────────────────────────────
    if choice == "1":
        q = input("  Query: ").strip()
        if not q:
            print("  Empty query.")
            continue
        n = _get_int("  Top N [default 10]: ") or 10
        keyword_search(q, n)

    # ── 2. BM25 similar ───────────────────────────────────────────────────────
    elif choice == "2":
        bid = input("  Business ID: ").strip()
        n   = _get_int("  Top N [default 10]: ") or 10
        bm25_similar(bid, n)

    # ── 3. Embedding similar ──────────────────────────────────────────────────
    elif choice == "3":
        bid = input("  Business ID: ").strip()
        n   = _get_int("  Top N [default 10]: ") or 10
        embedding_similar(bid, n)

    # ── 4. Hybrid similar ─────────────────────────────────────────────────────
    elif choice == "4":
        bid = input("  Business ID: ").strip()
        n   = _get_int("  Top N [default 10]: ") or 10
        a   = _get_float("  Alpha [0=pure embedding, 1=pure BM25, default 0.5]: ")
        if a is None:
            a = 0.5
        hybrid_similar(bid, n, alpha=a)

    # ── 5. Add / update business ──────────────────────────────────────────────
    elif choice == "5":
        bid   = input("  Business ID (new or existing): ").strip()
        bname = input("  Business name: ").strip()
        city  = input("  City [Philadelphia]: ").strip() or "Philadelphia"
        cats  = input("  Categories: ").strip()
        rev   = input("  New review text: ").strip()
        try:
            stars = float(input("  Stars (0-5): ") or 0)
        except ValueError:
            stars = 0.0
        add_or_update_business(bid, bname, city, cats, rev, stars=stars)

    # ── 6. Exit ───────────────────────────────────────────────────────────────
    elif choice == "6":
        print("\nGoodbye!")
        break

    else:
        print("  Invalid choice. Enter 1-6.")


╔══════════════════════════════════════════╗
║       YELP PHILADELPHIA SEARCH ENGINE   ║
╠══════════════════════════════════════════╣
║  1. Keyword search (BM25F + fuzzy)      ║
║  2. Similar reviews – BM25 keywords     ║
║  3. Similar reviews – Embeddings        ║
║  4. Similar reviews – Hybrid            ║
║  5. Add / update business (online sim)  ║
║  6. Exit                                ║
╚══════════════════════════════════════════╝



Choice [1-6]:  6



Goodbye!


## Cell 9 — Sample queries & discussion

In [10]:
# ── Sample keyword searches ───────────────────────────────────────────────────
print("\n" + "="*60)
print("SAMPLE QUERY 1: 'Italian restaurant'")
print("Expected: businesses with 'Italian' or 'restaurant' in name/category")
keyword_search("Italian restaurant", top_n=5)

print("\nSAMPLE QUERY 2: 'coffe' (intentional typo)")
print("Expected: fuzzy match finds coffee shops despite typo")
keyword_search("coffe", top_n=5)

print("\nSAMPLE QUERY 3: 'sushi bar'")
keyword_search("sushi bar", top_n=5)


SAMPLE QUERY 1: 'Italian restaurant'
Expected: businesses with 'Italian' or 'restaurant' in name/category


/var/folders/h4/d64f36gj5b32l9tlnmzrbh8c0000gn/T/ipykernel_24548/691816151.py:18: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


  Synonyms:    'italian restaurant'  →  '(italian OR pasta OR pizza OR trattoria OR osteria) (restaurant OR eatery OR dining OR bistro OR grill)'

════════════════════════════════════════════════════════════
  BM25 KEYWORD SEARCH
  Query   : (italian OR pasta OR pizza OR trattoria OR osteria) (restaurant OR eatery OR dining OR bistro OR grill)
  Hits    : 73 (of 585 estimated)
  Time    : 13.5 ms
════════════════════════════════════════════════════════════

  Rank 1  │  Score: 32.5965  │  DocID: V0DE3adlCc6m3t-yTxAg4g
  Business : Venice Pizza and Italian Eatery
  City     : Philadelphia
  Stars    : 0.0  │  Reviews: 0
  Category : 
  Snippet  : Venice **Pizza** and **Italian** **Eatery**
────────────────────────────────────────────────────────────

  Rank 2  │  Score: 27.6260  │  DocID: IrJbptQgdCXC4avPQKeChw
  Business : Italian Bistro
  City     : Philadelphia
  Stars    : 0.0  │  Reviews: 0
  Category : 
  Snippet  : **Italian** **Bistro**
──────────────────────────────────────────

In [11]:
# ── Sample similar-review searches ───────────────────────────────────────────
# Use a known business ID from the dataset
sample_id = df_index["business_id"].iloc[0]
print(f"Using sample business: {df_index.loc[0, 'name']} (ID: {sample_id})")

print("\nBM25 Similar:")
bm25_similar(sample_id, top_n=5)

print("\nEmbedding Similar:")
embedding_similar(sample_id, top_n=5)

print("\nHybrid Similar (alpha=0.4):")
hybrid_similar(sample_id, top_n=5, alpha=0.4)

Using sample business: St Honore Pastries (ID: MTSW4McQd7CbVtyjqoe9mw)

BM25 Similar:

════════════════════════════════════════════════════════════
  BM25 SIMILAR BUSINESSES
  Source  : St Honore Pastries  (ID: MTSW4McQd7CbVtyjqoe9mw)
  Keywords: egg cake buns tarts tea good like chinatown pastries bakery also get place sweet …
  Time    : 588.4 ms
════════════════════════════════════════════════════════════

  Rank 1  │  Score: 111.3772  │  DocID: uIFRFef7l43YxA8G3dU6Tg
  Business : Mayflower Bakery & Cafe
  City     : Philadelphia
  Stars    : 0.0  │  Reviews: 0
  Category : 
  Snippet  : **Pork** **Buns**. The Bubble **Tea** are made **fresh** every **time** you ordered it. **Also**, this **bakery** have a lots of little...many I want. In all **one** of my favorite **bakeries** in **Chinatown**.

They **also** have yummy cream **buns
────────────────────────────────────────────────────────────

  Rank 2  │  Score: 111.2294  │  DocID: ZRb203peHsQKibqyKqZaGg
  Business : KC's Pastries

In [12]:
# ── Online update demo ────────────────────────────────────────────────────────
print("Simulating online review stream: adding a brand-new business …")
add_or_update_business(
    business_id  = "TEST_BIZ_001",
    name         = "Test Ramen House",
    city         = "Philadelphia",
    categories   = "Ramen, Japanese, Noodles",
    new_reviews  = "Amazing tonkotsu broth. Very cosy atmosphere. Great service.",
    stars        = 4.5,
    review_count = 1,
)

print("\nNow searching for the newly added business:")
keyword_search("Test Ramen House", top_n=3)

print("\nAdding a second review to the same business:")
add_or_update_business(
    business_id  = "TEST_BIZ_001",
    name         = "Test Ramen House",
    city         = "Philadelphia",
    categories   = "Ramen, Japanese, Noodles",
    new_reviews  = "Came back again — the spicy miso is incredible!",
    stars        = 4.5,
    review_count = 2,
)

Simulating online review stream: adding a brand-new business …
Adding new business: Test Ramen House
✓ Index and embeddings updated for business_id=TEST_BIZ_001
  Total documents: 14570 BM25  |  14570 embeddings

Now searching for the newly added business:

════════════════════════════════════════════════════════════
  BM25 KEYWORD SEARCH
  Query   : test ramen house
  Hits    : 1 (of 4 estimated)
  Time    : 5.1 ms
════════════════════════════════════════════════════════════

  Rank 1  │  Score: 41.6084  │  DocID: TEST_BIZ_001
  Business : Test Ramen House
  City     : Philadelphia
  Stars    : 4.5  │  Reviews: 1
  Category : Ramen, Japanese, Noodles
  Snippet  : **Test** **Ramen** **House**
────────────────────────────────────────────────────────────

Adding a second review to the same business:
Updating existing business: Test Ramen House
✓ Index and embeddings updated for business_id=TEST_BIZ_001
  Total documents: 14570 BM25  |  14570 embeddings
